# PyNRPF v0.1.0 — Reproduce Key Numbers (Chunk 1 Skeleton, PySpark)

This notebook:
1. reads config from `configs/run.yaml`
2. downloads `rpf_dataset.parquet` from Zenodo
3. loads as a Spark DataFrame
4. runs minimal validation + time-based split scaffolding

In [ ]:
# Set up packages and spark

from __future__ import annotations

import json
import os
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional

import yaml  # pip install pyyaml
import requests  # pip install requests

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("PyNRPF_v0.1.0")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark:", spark.version)


In [ ]:
# Load config

REPO_ROOT = Path("..").resolve()  # notebook in /notebooks
CFG_PATH = REPO_ROOT / "configs" / "run.yaml"

if not CFG_PATH.exists():
    print(f"Config not found: {CFG_PATH}")
    raise SystemExit(0)

with CFG_PATH.open("r", encoding="utf-8") as f:
    cfg: Dict[str, Any] = yaml.safe_load(f)

def _get(path: str, default: Any = None) -> Any:
    cur: Any = cfg
    for p in path.split("."):
        if not isinstance(cur, dict) or p not in cur:
            return default
        cur = cur[p]
    return cur

def _req(path: str) -> Any:
    v = _get(path, default=None)
    if v is None:
        raise KeyError(f"Missing required config key: '{path}'")
    return v

RUN_TAG = str(_req("run.run_tag"))
OUTPUT_DIR = (REPO_ROOT / str(_req("paths.output_dir"))).resolve()
DATA_DIR = (REPO_ROOT / str(_req("paths.data_dir"))).resolve()
LOCAL_PARQUET = (DATA_DIR / str(_req("paths.local_parquet_filename"))).resolve()

ZENODO_RECORD_ID = str(_get("zenodo.record_id", "")).strip()
ZENODO_FILENAME = str(_req("zenodo.filename"))
ZENODO_TOKEN = str(_get("zenodo.access_token", "")).strip()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_TAG:", RUN_TAG)
print("DATA_DIR:", DATA_DIR)
print("LOCAL_PARQUET:", LOCAL_PARQUET)
print("ZENODO_RECORD_ID:", ZENODO_RECORD_ID if ZENODO_RECORD_ID else "<not set>")
print("ZENODO_FILENAME:", ZENODO_FILENAME)


In [ ]:
# Download dataset from zenodo
def _http_get(url: str, headers: Dict[str, str], timeout: int = 60, retries: int = 4) -> requests.Response:
    last_err: Optional[Exception] = None
    for i in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            if r.status_code >= 500:
                raise RuntimeError(f"HTTP {r.status_code} from {url}")
            return r
        except Exception as e:
            last_err = e
            time.sleep(1.5 * (i + 1))
    raise RuntimeError(f"Failed GET after {retries} tries: {url} | last_err={last_err}")

def ensure_parquet_local() -> Optional[Path]:
    if LOCAL_PARQUET.exists():
        return LOCAL_PARQUET

    if not ZENODO_RECORD_ID:
        print("Dataset not found locally, and zenodo.record_id is not set.")
        print("Set configs/run.yaml -> zenodo.record_id to your Zenodo record number.")
        return None

    api_url = f"https://zenodo.org/api/records/{ZENODO_RECORD_ID}"
    headers = {"User-Agent": "PyNRPF/0.1.0 (reproducibility notebook)"}
    if ZENODO_TOKEN:
        headers["Authorization"] = f"Bearer {ZENODO_TOKEN}"

    print("Fetching Zenodo record:", api_url)
    r = _http_get(api_url, headers=headers)
    if r.status_code != 200:
        print("Failed to fetch Zenodo record.")
        print("HTTP:", r.status_code)
        print("Body:", (r.text or "")[:500])
        return None

    meta = r.json()
    files = meta.get("files", []) or []
    if not files:
        print("No files found in Zenodo record.")
        return None

    target = None
    for f in files:
        key = f.get("key") or f.get("filename") or f.get("name")
        if key == ZENODO_FILENAME:
            target = f
            break

    if target is None:
        available = [x.get("key") or x.get("filename") or x.get("name") for x in files]
        print("Target file not found in record:", ZENODO_FILENAME)
        print("Available files:", available)
        return None

    links = target.get("links", {}) or {}
    dl_url = links.get("self") or links.get("download")
    if not dl_url:
        print("No usable download link found in file metadata (expected links.self or links.download).")
        print("links:", links)
        return None

    tmp_path = LOCAL_PARQUET.with_suffix(".parquet.part")
    print("Downloading:", dl_url)
    with requests.get(dl_url, headers=headers, stream=True, timeout=120) as resp:
        if resp.status_code != 200:
            print("Download failed:", resp.status_code)
            print("Body:", (resp.text or "")[:300])
            return None
        with open(tmp_path, "wb") as fp:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    fp.write(chunk)

    tmp_path.replace(LOCAL_PARQUET)
    print("Saved:", LOCAL_PARQUET)
    return LOCAL_PARQUET

local_path = ensure_parquet_local()
if local_path is None:
    print("Stopping (no local dataset available yet).")
    raise SystemExit(0)

In [ ]:
# Load dataset

df = spark.read.parquet(str(local_path))
print("Loaded Spark DF")
print("Rows:", df.count())
df.printSchema()
df.show(5, truncate=False)

In [ ]:
# Minimal validation

COL_SITE = str(_req("columns.site"))
COL_TS = str(_req("columns.timestamp"))
COL_NET = str(_req("columns.net_load_mw"))
COL_SOLAR = str(_req("columns.solar_mw"))
COL_GT = str(_req("columns.net_load_ground_truth"))

required = [COL_SITE, COL_TS, COL_NET, COL_SOLAR, COL_GT]
missing = [c for c in required if c not in df.columns]
if missing:
    print("Missing required columns:", missing)
    print("Fix configs/run.yaml -> columns mapping.")
    raise SystemExit(0)

df2 = df.withColumn(COL_TS, F.to_timestamp(F.col(COL_TS)))
bad_ts = df2.filter(F.col(COL_TS).isNull()).count()
if bad_ts:
    print(f"Unparsable timestamps in '{COL_TS}': {bad_ts} rows")
    raise SystemExit(0)

dup = (
    df2.groupBy(COL_SITE, COL_TS)
       .count()
       .filter(F.col("count") > 1)
       .count()
)
if dup:
    print("Duplicate (site,timestamp) keys found:", dup)
    raise SystemExit(0)

miss = {c: df2.filter(F.col(c).isNull()).count() for c in required}
print("Missingness:", miss)

df = df2


In [ ]:
# Train test split

TRAIN_START = str(_req("split.train_start"))
TRAIN_END   = str(_req("split.train_end"))
TEST_START  = str(_req("split.test_start"))
TEST_END    = str(_req("split.test_end"))

train_df = df.filter(
    (F.col(COL_TS) >= F.to_timestamp(F.lit(TRAIN_START))) &
    (F.col(COL_TS) <= F.to_timestamp(F.lit(TRAIN_END)))
)
test_df = df.filter(
    (F.col(COL_TS) >= F.to_timestamp(F.lit(TEST_START))) &
    (F.col(COL_TS) <= F.to_timestamp(F.lit(TEST_END)))
)

print("Train rows:", train_df.count(), "|", TRAIN_START, "to", TRAIN_END)
print("Test rows:",  test_df.count(),  "|", TEST_START,  "to", TEST_END)


In [ ]:
# Day level table

day_df = (
    df.withColumn("date", F.to_date(F.col(COL_TS)))
      .groupBy(COL_SITE, "date")
      .agg(
          F.count("*").alias("n_intervals"),
          F.max((F.col(COL_GT) < F.lit(0)).cast("int")).alias("any_gt_neg"),
      )
)

print("Day rows:", day_df.count())
day_df.show(5, truncate=False)


In [ ]:
# m7_threshold

In [ ]:
# m8_xgb

In [ ]:
# evaluation & exports
METRICS_OUT = OUTPUT_DIR / f"metrics__{RUN_TAG}.json"

metrics = {
    "run_tag": RUN_TAG,
    "generated_at_utc": datetime.utcnow().isoformat() + "Z",
    "zenodo_record_id": ZENODO_RECORD_ID,
    "zenodo_filename": ZENODO_FILENAME,
    "local_parquet": str(local_path),
    "split": {
        "train_start": TRAIN_START,
        "train_end": TRAIN_END,
        "test_start": TEST_START,
        "test_end": TEST_END,
    },
    "status": "chunk_1_skeleton_only",
    "todo": [
        "Implement m7_threshold (Chunk 5)",
        "Implement m8_xgb + features (Chunk 6)",
        "Implement eval outputs (Chunk 7)",
    ],
}

with METRICS_OUT.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Wrote:", METRICS_OUT)
